# VSL400 – Trích xuất landmark toàn thân bằng MediaPipe Holistic

- Dataset đầu vào: `/kaggle/input/datasets/bowboochua9/vsl400`
- Kết quả trung gian: `/kaggle/working/vsl400_holistic/`
- File ZIP cuối cùng: `/kaggle/working/vsl400_holistic_results.zip`
- Vector mỗi frame: **225D** = Right Hand 63D + Left Hand 63D + Pose 99D.
- Resume: chỉ bỏ qua video khi **cả file NPY hợp lệ và dòng CSV có trạng thái hoàn tất**.

## 1. Cài đặt và kiểm tra thư viện

In [1]:
import os
import sys
import subprocess
import importlib
import warnings
from pathlib import Path
from urllib.request import urlretrieve

# ============================================================
# 1. Hạn chế log không cần thiết
# ============================================================

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"] = "3"
os.environ["GLOG_logtostderr"] = "0"
os.environ["GLOG_stderrthreshold"] = "3"
os.environ["ABSL_LOG_PREFIX"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

warnings.filterwarnings("ignore")

# ============================================================
# 2. Cài đặt thư viện
# ============================================================

def install_dependencies():
    print("[SETUP] Đang cài đặt MediaPipe và TQDM...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mediapipe==0.10.35",
        "tqdm"
    ])

    importlib.invalidate_caches()


# ============================================================
# 3. Import thư viện
# ============================================================

def import_mediapipe_stack():
    import cv2
    import mediapipe as mp
    import numpy as np
    from tqdm.auto import tqdm

    if not hasattr(mp, "tasks"):
        raise ImportError("MediaPipe không có Tasks API.")

    if not hasattr(mp.tasks.vision, "HolisticLandmarker"):
        raise ImportError(
            "Không tìm thấy HolisticLandmarker trong MediaPipe."
        )

    return cv2, mp, np, tqdm


try:
    cv2, mp, np, tqdm = import_mediapipe_stack()

except Exception as import_error:
    print("[SETUP] MediaPipe chưa sẵn sàng:", repr(import_error))

    install_dependencies()

    cv2, mp, np, tqdm = import_mediapipe_stack()


# ============================================================
# 4. Tải model Holistic Landmarker
# ============================================================

MODEL_PATH = Path("/kaggle/working/holistic_landmarker.task")

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "holistic_landmarker/holistic_landmarker/float16/1/"
    "holistic_landmarker.task"
)

if not MODEL_PATH.exists():
    print("[MODEL] Đang tải Holistic Landmarker...")

    urlretrieve(
        MODEL_URL,
        str(MODEL_PATH)
    )

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size == 0:
    raise RuntimeError("Không tải được model Holistic Landmarker.")

# ============================================================
# 5. Kiểm tra môi trường
# ============================================================

print("=" * 60)
print("KIỂM TRA MÔI TRƯỜNG")
print("=" * 60)

print("Python      :", sys.version.split()[0])
print("MediaPipe   :", mp.__version__)
print("OpenCV      :", cv2.__version__)
print("NumPy       :", np.__version__)
print("Model       :", MODEL_PATH)
print("Model size  :", round(MODEL_PATH.stat().st_size / 1024 / 1024, 2), "MB")

print("=" * 60)
print("[OK] Môi trường đã sẵn sàng.")

[SETUP] MediaPipe chưa sẵn sàng: ModuleNotFoundError("No module named 'mediapipe'")
[SETUP] Đang cài đặt MediaPipe và TQDM...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


[MODEL] Đang tải Holistic Landmarker...
KIỂM TRA MÔI TRƯỜNG
Python      : 3.13.15
MediaPipe   : 0.10.35
OpenCV      : 4.14.0
NumPy       : 2.1.3
Model       : /kaggle/working/holistic_landmarker.task
Model size  : 13.05 MB
[OK] Môi trường đã sẵn sàng.


## 2. Cấu hình đường dẫn và tham số

In [2]:
from pathlib import Path
import shutil
import zipfile
import time
import csv
import json
import os
import sys

DATASET_ROOT = Path("/kaggle/input/datasets/bowboochua9/vsl400")
WORK_ROOT = Path("/kaggle/working/vsl400_holistic")
OUTPUT_DIR = WORK_ROOT / "data" / "processed" / "landmarks_holistic"
REPORT_DIR = WORK_ROOT / "reports"
ZIP_PATH = Path("/kaggle/working/vsl400_holistic_results.zip")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Khong tim thay dataset: {DATASET_ROOT}. "
        "Hay kiem tra Add Input va slug dataset tren Kaggle."
    )

WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# CẤU HÌNH PART
PARTS_TO_RUN = [5]

NUM_WORKERS = 2
VIEWS_TO_RUN = ["front_view", "left_view", "right_view"]
DRY_RUN = False
MAX_VIDEOS_PER_VIEW = None  # None = toan bo

if NUM_WORKERS < 1:
    raise ValueError("NUM_WORKERS phai >= 1")
if not set(PARTS_TO_RUN).issubset(set(range(1, 8))):
    raise ValueError("PARTS_TO_RUN chi duoc chua cac gia tri tu 1 den 7")

print("Dataset dau vao :", DATASET_ROOT)
print("Thu muc NPY     :", OUTPUT_DIR)
print("Thu muc bao cao :", REPORT_DIR)
print("Workers         :", NUM_WORKERS)
print("Parts           :", PARTS_TO_RUN)
print("Views           :", VIEWS_TO_RUN)
print("Dry run         :", DRY_RUN)
print("Python          :", sys.version.split()[0])

Dataset dau vao : /kaggle/input/datasets/bowboochua9/vsl400
Thu muc NPY     : /kaggle/working/vsl400_holistic/data/processed/landmarks_holistic
Thu muc bao cao : /kaggle/working/vsl400_holistic/reports
Workers         : 2
Parts           : [5]
Views           : ['front_view', 'left_view', 'right_view']
Dry run         : False
Python          : 3.13.15


## 3. Kiểm tra cấu trúc dataset

In [3]:
print("KIEM TRA DATASET VSL400")
print("=" * 80)

missing_items = []
for part_idx in range(1, 8):
    split_dir = DATASET_ROOT / f"Part_{part_idx}" / f"split_{part_idx}"
    if not split_dir.exists():
        missing_items.append(str(split_dir))
        print(f"[THIEU] Part_{part_idx}: {split_dir}")
        continue

    for view in VIEWS_TO_RUN:
        video_dir = split_dir / view
        json_path = split_dir / f"{view}.json"
        video_count = len(list(video_dir.glob("*.mp4"))) if video_dir.exists() else 0
        json_ok = json_path.is_file()
        print(
            f"Part_{part_idx:>1} | {view:<12} | "
            f"videos={video_count:>6,} | metadata={'OK' if json_ok else 'MISSING'}"
        )
        if not json_ok:
            missing_items.append(str(json_path))

if missing_items:
    print("\n[CANH BAO] Mot so duong dan / metadata bi thieu. Cac view nay se duoc bo qua.")
else:
    print("\n[OK] Da tim thay day du thu muc va metadata cho cac Part/Views da chon.")

KIEM TRA DATASET VSL400
Part_1 | front_view   | videos= 3,537 | metadata=OK
Part_1 | left_view    | videos= 3,537 | metadata=OK
Part_1 | right_view   | videos= 3,537 | metadata=OK
Part_2 | front_view   | videos= 3,536 | metadata=OK
Part_2 | left_view    | videos= 3,536 | metadata=OK
Part_2 | right_view   | videos= 3,536 | metadata=OK
Part_3 | front_view   | videos= 3,536 | metadata=OK
Part_3 | left_view    | videos= 3,536 | metadata=OK
Part_3 | right_view   | videos= 3,536 | metadata=OK
Part_4 | front_view   | videos= 3,536 | metadata=OK
Part_4 | left_view    | videos= 3,536 | metadata=OK
Part_4 | right_view   | videos= 3,536 | metadata=OK
Part_5 | front_view   | videos= 3,536 | metadata=OK
Part_5 | left_view    | videos= 3,536 | metadata=OK
Part_5 | right_view   | videos= 3,536 | metadata=OK
Part_6 | front_view   | videos= 3,536 | metadata=OK
Part_6 | left_view    | videos= 3,536 | metadata=OK
Part_6 | right_view   | videos= 3,536 | metadata=OK
Part_7 | front_view   | videos= 3,536 | 

## 4. Toàn bộ module trích xuất Holistic

In [4]:
import os
import sys
import json
import csv
import time
import argparse
import warnings

from pathlib import Path
from contextlib import contextmanager
from concurrent.futures import ProcessPoolExecutor, as_completed

warnings.filterwarnings("ignore")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"] = "3"
os.environ["GLOG_logtostderr"] = "0"
os.environ["GLOG_stderrthreshold"] = "3"
os.environ["ABSL_LOG_PREFIX"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

@contextmanager
def suppress_c_stderr():
    null_fd = None
    old_stderr_fd = None
    try:
        null_fd = os.open(os.devnull, os.O_RDWR)
        old_stderr_fd = os.dup(2)
        os.dup2(null_fd, 2)
        yield
    finally:
        if old_stderr_fd is not None:
            os.dup2(old_stderr_fd, 2)
            os.close(old_stderr_fd)
        if null_fd is not None:
            os.close(null_fd)

with suppress_c_stderr():
    import cv2
    import mediapipe as mp
    import numpy as np

from tqdm.auto import tqdm

if not hasattr(mp.tasks.vision, "HolisticLandmarker"):
    raise RuntimeError("MediaPipe hiện tại không hỗ trợ HolisticLandmarker.")

if "MODEL_PATH" not in globals():
    MODEL_PATH = Path("/kaggle/working/holistic_landmarker.task")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy model: {MODEL_PATH}.")

PROJECT_ROOT = Path("/kaggle/working/vsl400_holistic")
RAW_DATA_DIR = Path("/kaggle/input/datasets/bowboochua9/vsl400")
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "landmarks_holistic"
REPORT_DIR = PROJECT_ROOT / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

VIEWS = ["front_view", "left_view", "right_view"]
NUM_PARTS = 7
COMPLETED_STATUSES = {"ok", "no_hand", "no_pose"}
CSV_FIELDS = [
    "video_id", "signer_id", "gloss", "view", "part", 
    "num_frames", "right_hand_det_rate", "left_hand_det_rate", 
    "pose_det_rate", "status", "error_msg"
]

def get_part_report_path(part_idx: int) -> Path:
    return REPORT_DIR / f"extraction_report_holistic_Part_{part_idx}.csv"

def get_all_report_path() -> Path:
    return REPORT_DIR / "extraction_report_holistic_all.csv"

def task_key(data: dict) -> tuple:
    return (
        str(data.get("part", "")),
        str(data.get("view", "")),
        str(data.get("video_id", "")),
        str(data.get("signer_id", ""))
    )

def read_part_report(part_idx: int) -> list:
    report_file = get_part_report_path(part_idx)
    if not report_file.exists():
        return []
    try:
        with open(report_file, "r", newline="", encoding="utf-8-sig") as f:
            return list(csv.DictReader(f))
    except (OSError, csv.Error) as exc:
        print(f"[WARN] Không đọc được CSV Part_{part_idx}: {exc}")
        return []

def write_part_report(part_idx: int, rows: list) -> None:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    report_file = get_part_report_path(part_idx)
    temp_path = report_file.with_suffix(".csv.tmp")
    with open(temp_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp_path, report_file)

def get_tasks_for_part(part_idx: int, views_to_process: list = None, dry_run: bool = False, max_videos: int = None) -> list:
    if views_to_process is None:
        views_to_process = VIEWS
    split_dir = RAW_DATA_DIR / f"Part_{part_idx}" / f"split_{part_idx}"
    if not split_dir.exists():
        print(f"[WARN] Không tìm thấy thư mục: {split_dir}")
        return []

    tasks = []
    for view in views_to_process:
        video_dir = split_dir / view
        json_path = split_dir / f"{view}.json"
        output_dir = OUTPUT_DIR / f"Part_{part_idx}" / view
        output_dir.mkdir(parents=True, exist_ok=True)

        if not json_path.exists():
            continue

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                metadata_list = json.load(f)
        except (OSError, json.JSONDecodeError):
            continue

        view_count = 0
        for entry in metadata_list:
            if max_videos is not None and view_count >= max_videos:
                break
            video_id = str(entry["video_id"])
            signer_id = str(entry["signer_id"])
            gloss = entry.get("gloss", "")
            video_path = video_dir / f"{video_id}.mp4"
            output_path = output_dir / f"{video_id}_{signer_id}.npy"

            if not video_path.exists():
                continue

            tasks.append({
                "video_path": str(video_path),
                "output_path": str(output_path),
                "video_id": video_id,
                "signer_id": signer_id,
                "gloss": gloss,
                "view": view,
                "part": part_idx
            })
            view_count += 1

    if dry_run:
        tasks = tasks[:10]
    return tasks

def get_pending_tasks(tasks: list, report_rows: list) -> list:
    successful_csv_keys = {task_key(row) for row in report_rows if row.get("status") in COMPLETED_STATUSES}
    pending = []
    skipped = 0

    for task in tasks:
        key = task_key(task)
        output_file = Path(task["output_path"])
        npy_valid = False
        if output_file.is_file():
            try:
                arr = np.load(output_file, mmap_mode="r")
                npy_valid = (arr.ndim == 2 and arr.shape[1] == 225 and arr.shape[0] > 0)
                del arr
            except Exception:
                npy_valid = False

        csv_exists = (key in successful_csv_keys)
        if npy_valid and csv_exists:
            skipped += 1
        else:
            pending.append(task)

    print(f"[RESUME] Bỏ qua: {skipped} video | Cần xử lý: {len(pending)} video")
    return pending

def process_video_holistic(task: dict) -> dict:
    video_path = task["video_path"]
    output_path = task["output_path"]
    result = {
        "video_id": task["video_id"], "signer_id": task["signer_id"],
        "gloss": task["gloss"], "view": task["view"], "part": task["part"],
        "num_frames": 0, "right_hand_det_rate": 0.0, "left_hand_det_rate": 0.0,
        "pose_det_rate": 0.0, "status": "error", "error_msg": ""
    }

    cap = None
    holistic = None
    try:
        with suppress_c_stderr():
            options = mp.tasks.vision.HolisticLandmarkerOptions(
                base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
                running_mode=mp.tasks.vision.RunningMode.VIDEO,
                min_face_detection_confidence=0.5, min_face_landmarks_confidence=0.5,
                min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
                min_hand_landmarks_confidence=0.5, output_face_blendshapes=False,
                output_segmentation_mask=False
            )
            holistic = mp.tasks.vision.HolisticLandmarker.create_from_options(options)

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            result["error_msg"] = "Không mở được video"
            return result

        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0 or not np.isfinite(fps):
            fps = 30.0

        all_frames = []
        frame_count = 0
        rh_count, lh_count, pose_count = 0, 0, 0
        last_timestamp_ms = -1

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            timestamp_ms = int(round(frame_count * 1000.0 / fps))
            timestamp_ms = max(timestamp_ms, last_timestamp_ms + 1)
            last_timestamp_ms = timestamp_ms

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            with suppress_c_stderr():
                res = holistic.detect_for_video(mp_image, timestamp_ms)

            arr = np.zeros(225, dtype=np.float32)
            if res.right_hand_landmarks:
                rh_count += 1
                for i, lm in enumerate(res.right_hand_landmarks[:21]):
                    arr[i * 3 : i * 3 + 3] = [lm.x, lm.y, lm.z]

            if res.left_hand_landmarks:
                lh_count += 1
                for i, lm in enumerate(res.left_hand_landmarks[:21]):
                    arr[63 + i * 3 : 63 + i * 3 + 3] = [lm.x, lm.y, lm.z]

            if res.pose_landmarks:
                pose_count += 1
                for i, lm in enumerate(res.pose_landmarks[:33]):
                    arr[126 + i * 3 : 126 + i * 3 + 3] = [lm.x, lm.y, lm.z]

            all_frames.append(arr)
            frame_count += 1

        if frame_count == 0:
            result["error_msg"] = "Video rỗng (0 frame)"
            return result

        landmarks_array = np.asarray(all_frames, dtype=np.float32)
        output_file = Path(output_path)
        temp_output = output_file.with_name(output_file.stem + ".tmp.npy")
        np.save(temp_output, landmarks_array)
        os.replace(temp_output, output_file)

        result["num_frames"] = frame_count
        result["right_hand_det_rate"] = round(rh_count / frame_count * 100, 2)
        result["left_hand_det_rate"] = round(lh_count / frame_count * 100, 2)
        result["pose_det_rate"] = round(pose_count / frame_count * 100, 2)

        has_hand = (rh_count > 0 or lh_count > 0)
        has_pose = (pose_count > 0)

        if has_hand and has_pose:
            result["status"] = "ok"
        elif not has_hand and has_pose:
            result["status"] = "no_hand"
            result["error_msg"] = "Không phát hiện bàn tay"
        elif has_hand and not has_pose:
            result["status"] = "no_pose"
            result["error_msg"] = "Không phát hiện Pose"
        else:
            result["status"] = "no_hand"
            result["error_msg"] = "Không phát hiện bàn tay và Pose"

    except Exception as exc:
        result["status"] = "error"
        result["error_msg"] = str(exc)
    finally:
        if cap is not None:
            cap.release()
        if holistic is not None:
            try:
                holistic.close()
            except Exception:
                pass
    return result

def process_part(part_idx: int, num_workers: int, views: list, dry_run: bool = False, max_videos: int = None) -> None:
    print("\n" + "=" * 75)
    print(f"BẮT ĐẦU TRÍCH XUẤT HOLISTIC: Part_{part_idx}")
    print("=" * 75)

    tasks = get_tasks_for_part(part_idx, views_to_process=views, dry_run=dry_run, max_videos=max_videos)
    if not tasks:
        return

    existing_rows = read_part_report(part_idx)
    pending_tasks = get_pending_tasks(tasks, existing_rows)
    if not pending_tasks:
        print(f"[XONG] Part_{part_idx} đã hoàn tất.")
        return

    results_map = {task_key(row): row for row in existing_rows}
    t0 = time.time()
    completed_in_run = 0

    try:
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            future_to_task = {executor.submit(process_video_holistic, task): task for task in pending_tasks}
            pbar = tqdm(as_completed(future_to_task), total=len(pending_tasks), desc=f"Part_{part_idx} [Holistic]", unit="video", ncols=90)
            
            for future in pbar:
                task = future_to_task[future]
                try:
                    res = future.result()
                    results_map[task_key(res)] = res
                    completed_in_run += 1
                except Exception as exc:
                    err_res = {
                        "video_id": task["video_id"], "signer_id": task["signer_id"],
                        "gloss": task["gloss"], "view": task["view"], "part": task["part"],
                        "num_frames": 0, "right_hand_det_rate": 0.0, "left_hand_det_rate": 0.0,
                        "pose_det_rate": 0.0, "status": "error", "error_msg": str(exc)
                    }
                    results_map[task_key(err_res)] = err_res
                    completed_in_run += 1

                if completed_in_run % 50 == 0:
                    ordered_rows = [results_map[task_key(t)] for t in tasks if task_key(t) in results_map]
                    write_part_report(part_idx, ordered_rows)
            pbar.close()
    finally:
        ordered_rows = [results_map[task_key(t)] for t in tasks if task_key(t) in results_map]
        write_part_report(part_idx, ordered_rows)
        elapsed = time.time() - t0
        print(f"\n[HOÀN TẤT] Part_{part_idx}: Xử lý {completed_in_run} video trong {elapsed:.1f}s")

print("[OK] Đã nạp module Holistic Extractor.")

[OK] Đã nạp module Holistic Extractor.


## 5. Chạy tự động các Part theo cấu hình

In [5]:
# Tự động duyệt qua danh sách PARTS_TO_RUN đã định nghĩa ở Cell 2
for part_idx in PARTS_TO_RUN:
    process_part(
        part_idx,
        num_workers=NUM_WORKERS,
        views=VIEWS_TO_RUN,
        dry_run=DRY_RUN,
        max_videos=MAX_VIDEOS_PER_VIEW
    )


BẮT ĐẦU TRÍCH XUẤT HOLISTIC: Part_5
[RESUME] Bỏ qua: 0 video | Cần xử lý: 10608 video


Part_5 [Holistic]:   0%|                                     | 0/10608 [00:00<?, ?video/s]


[HOÀN TẤT] Part_5: Xử lý 10608 video trong 30295.3s


## 6. Tổng hợp báo cáo phần tài khoản này

In [6]:
import csv

def merge_partial_reports():
    all_rows = []
    for part_idx in PARTS_TO_RUN:
        report_file = REPORT_DIR / f"extraction_report_holistic_Part_{part_idx}.csv"
        if report_file.exists():
            with open(report_file, "r", encoding="utf-8-sig") as f:
                all_rows.extend(list(csv.DictReader(f)))
            print(f"Đã nạp báo cáo: {report_file.name}")
        else:
            print(f"[CẢNH BÁO] Chưa có báo cáo của Part_{part_idx}")

    if all_rows:
        all_path = REPORT_DIR / "extraction_report_holistic_all.csv"
        with open(all_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\n[XONG] Đã gộp {len(all_rows)} dòng vào: {all_path.name}")
    else:
        print("[CẢNH BÁO] Không có dữ liệu báo cáo nào để tổng hợp.")

merge_partial_reports()

Đã nạp báo cáo: extraction_report_holistic_Part_5.csv

[XONG] Đã gộp 10608 dòng vào: extraction_report_holistic_all.csv


## 7. Thống kê và phân tích chất lượng dữ liệu

In [7]:
import pandas as pd

report_path = REPORT_DIR / "extraction_report_holistic_all.csv"
if report_path.exists():
    df = pd.read_csv(report_path)
    print(f"Tổng số video có trong báo cáo: {len(df):,}")
    print("\nPhân phối trạng thái xử lý:")
    display(df["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
else:
    print("Chưa tìm thấy báo cáo tổng hợp.")

Tổng số video có trong báo cáo: 10,608

Phân phối trạng thái xử lý:


,count
status,
ok,10608


## 8. Đóng gói kết quả thành file ZIP

In [8]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

files_to_zip = []
for folder in (OUTPUT_DIR, REPORT_DIR):
    if folder.exists():
        files_to_zip.extend(
            p for p in folder.rglob("*")
            if p.is_file() and not p.name.endswith((".tmp", ".tmp.npy"))
        )

if not files_to_zip:
    raise RuntimeError("Không có file kết quả để đóng gói.")

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as zf:
    for file_path in sorted(files_to_zip):
        zf.write(file_path, arcname=file_path.relative_to(WORK_ROOT))

zip_size_gb = ZIP_PATH.stat().st_size / (1024 ** 3)
print("=" * 70)
print("ĐÃ TẠO FILE ZIP THÀNH CÔNG")
print("Đường dẫn :", ZIP_PATH)
print(f"Số file   : {len(files_to_zip):,}")
print(f"Dung lượng: {zip_size_gb:.3f} GB")
print("=" * 70)

ĐÃ TẠO FILE ZIP THÀNH CÔNG
Đường dẫn : /kaggle/working/vsl400_holistic_results.zip
Số file   : 10,610
Dung lượng: 0.479 GB
